# Building a World-Aware Agent with Mistral

**Author:** [Patrick Liu](https://github.com/patrickliu0077), SimpleFunctions

LLMs have a knowledge cutoff. This cookbook shows how to give Mistral models **real-time world awareness** by injecting calibrated prediction market data — probabilities backed by real money from 9,706 contracts.

We use [SimpleFunctions](https://simplefunctions.dev/world) to inject ~800 tokens of structured world state (geopolitics, economy, energy, elections, crypto, tech) into the system prompt. No API key needed.

**What you'll build:**
1. System prompt injection with live world state
2. Function calling for market search and topic drilling
3. An agentic loop that uses prediction market data

## Setup

In [ ]:
%pip install mistralai requests -q

In [ ]:
import os
import json
import requests
from mistralai import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
MODEL = "mistral-large-latest"

## Step 1: Fetch the World State

One API call. ~800 tokens of markdown. Anchor contracts (recession, Fed rate, Iran invasion) always appear. Updated every 15 minutes from Kalshi (CFTC-regulated) and Polymarket.

In [ ]:
world = requests.get("https://simplefunctions.dev/api/agent/world").text
print(world)

## Step 2: World-Aware Mistral via System Prompt

In [ ]:
system_prompt = f"""You are a macro research analyst with real-time world awareness.

Use the data below as ground truth. Cite specific probabilities and prices.
Do not hallucinate numbers — if it's not in the data, say you don't know.

{world}"""

response = client.chat.complete(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What's the current geopolitical risk level and how does it affect energy markets?"},
    ],
)

print(response.choices[0].message.content)

## Step 3: Function Calling for Deeper Data

Define tools so Mistral can search specific markets and get focused world state.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_world_state",
            "description": "Get current world state from prediction markets. Focus on specific topics for deeper coverage with the same token budget.",
            "parameters": {
                "type": "object",
                "properties": {
                    "focus": {
                        "type": "string",
                        "description": "Comma-separated topics: geopolitics, economy, energy, elections, crypto, tech"
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_world_delta",
            "description": "Get only what changed in the world since a time. ~30-50 tokens vs 800 for full state.",
            "parameters": {
                "type": "object",
                "properties": {
                    "since": {"type": "string", "description": "Time window: 30m, 1h, 6h, 24h"}
                },
                "required": ["since"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_prediction_markets",
            "description": "Search prediction markets for specific contracts. Returns prices, volumes, spreads.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query, e.g. 'iran oil' or 'fed rate cut'"}
                },
                "required": ["query"]
            }
        }
    }
]


def execute_tool(name, args):
    if name == "get_world_state":
        url = "https://simplefunctions.dev/api/agent/world"
        if args.get("focus"):
            url += f"?focus={args['focus']}"
        return requests.get(url).text
    elif name == "get_world_delta":
        return requests.get(f"https://simplefunctions.dev/api/agent/world/delta?since={args['since']}").text
    elif name == "search_prediction_markets":
        resp = requests.get(f"https://simplefunctions.dev/api/public/scan?q={args['query']}&limit=10")
        return json.dumps(resp.json().get("markets", [])[:10], indent=2)
    return "Unknown tool"

In [ ]:
# Agentic loop with function calling
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Search for Iran-related contracts and analyze the risk of oil supply disruption."},
]

while True:
    response = client.chat.complete(
        model=MODEL,
        messages=messages,
        tools=tools,
    )

    msg = response.choices[0].message

    if msg.tool_calls:
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
            result = execute_tool(tc.function.name, args)
            print(f"Tool: {tc.function.name}({args})")
            messages.append({
                "role": "tool",
                "name": tc.function.name,
                "content": result,
                "tool_call_id": tc.id,
            })
    else:
        print("\n" + msg.content)
        break

## Why Prediction Markets?

| | Web Search | News API | Prediction Markets |
|---|---|---|---|
| **Output** | Narrative text | Headlines | Calibrated probabilities |
| **Token cost** | 2,000-5,000 | 500-1,000 | ~800 for everything |
| **Latency** | 2-5 seconds | 500ms | ~200ms (cached) |
| **Calibration** | None | None | Participants lose money when wrong |

A price of 53c on "Iran invasion" encodes the aggregate judgment of everyone with money at risk on that question.

**Links:**
- [World State API](https://simplefunctions.dev/api/agent/world) — try it now (no auth)
- [Documentation](https://simplefunctions.dev/docs/guide)
- [All endpoints](https://simplefunctions.dev/api/tools)